# Stage 10: Modeling & Evaluation (Crosswalk-Corrected 5-Election Panel)

**Pipeline Stage:** Modeling  
**Primary Input:** `data/processed/08_feature_engineered_dataset/ward_features_2000_2021_crosswalk_corrected.csv`  
**Output Directory:** `data/processed/10_model_outputs/`  

### Methodology & Evaluation Standard
- **Target:** Ward-level voter turnout rate (`TurnoutRate`) in held-out 2021.
- **Benchmark:** Naive historical baseline (`PreviousTurnout` from 2016).
- **Train/Test Split:** Time-respecting (train on 2006, 2011, 2016; test on 2021).
- **Context Feature Testing:** Municipality-level context features are tested before deciding whether to keep them.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'Modeling' else Path.cwd()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = PROCESSED_DIR / '10_model_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / '08_feature_engineered_dataset' / 'ward_features_2000_2021_crosswalk_corrected.csv')

train_mask = df['ElectionYear'].isin([2006, 2011, 2016]) & df['PreviousTurnout'].notna()
test_mask = (df['ElectionYear'] == 2021) & df['PreviousTurnout'].notna()

train = df[train_mask].copy()
test = df[test_mask].copy()

print(f'Training observations (2006, 2011, 2016): {len(train):,}')
print(f'Test observations (2021 held-out):        {len(test):,}')


### Evaluation 1: Naive Historical Persistence Baseline
Each ward's predicted 2021 turnout is its observed 2016 turnout (`PreviousTurnout`).

In [2]:
y_test = test['TurnoutRate']
y_pred_base = test['PreviousTurnout']

base_mae = mean_absolute_error(y_test, y_pred_base)
base_rmse = np.sqrt(mean_squared_error(y_test, y_pred_base))
base_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline MAE:  {base_mae:.4f} percentage points')
print(f'Baseline RMSE: {base_rmse:.4f} percentage points')
print(f'Baseline R²:   {base_r2:.4f}')


### Evaluation 2: Random Forest (Ward-Level History Features Only)

In [3]:
ward_features = ['PreviousTurnout', 'RegisteredVotersChange', 'RegistrationGrowth', 'SafeProvincialAverageTurnout', 'SafeBelowProvincialAverageTurnout']
X_train_w = train[ward_features].fillna(train[ward_features].median())
y_train = train['TurnoutRate']
X_test_w = test[ward_features].fillna(train[ward_features].median())

rf_w = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_w.fit(X_train_w, y_train)
y_pred_rf_w = rf_w.predict(X_test_w)

rf_w_mae = mean_absolute_error(y_test, y_pred_rf_w)
rf_w_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf_w))
rf_w_r2 = r2_score(y_test, y_pred_rf_w)

print(f'RF (Ward History) MAE:  {rf_w_mae:.4f} percentage points')
print(f'RF (Ward History) RMSE: {rf_w_rmse:.4f} percentage points')
print(f'RF (Ward History) R²:   {rf_w_r2:.4f}')
print(f'MAE Improvement over Baseline:  {(base_mae - rf_w_mae) / base_mae * 100:.2f}% ({(base_mae - rf_w_mae):.4f} pts)')
print(f'RMSE Improvement over Baseline: {(base_rmse - rf_w_rmse) / base_rmse * 100:.2f}% ({(base_rmse - rf_w_rmse):.4f} pts)')


### Evaluation 3: Testing Municipality Context Features
We test whether adding broad municipal demographic features improves the held-out test score.

In [4]:
muni_cols = ['Household', 'Homeless', 'Transient', 'Institution', 'UrbanArea', 'TribalOrTraditionalArea', 'FarmArea', 'MalePopulation', 'Population']
available_muni_cols = [c for c in muni_cols if c in train.columns]

X_train_all = train[ward_features + available_muni_cols].fillna(train[ward_features + available_muni_cols].median())
X_test_all = test[ward_features + available_muni_cols].fillna(train[ward_features + available_muni_cols].median())

rf_all = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_all.fit(X_train_all, y_train)
y_pred_rf_all = rf_all.predict(X_test_all)

rf_all_mae = mean_absolute_error(y_test, y_pred_rf_all)
rf_all_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf_all))

print(f'RF (+ Muni Context) MAE:  {rf_all_mae:.4f} percentage points')
print(f'RF (+ Muni Context) RMSE: {rf_all_rmse:.4f} percentage points')
print(f'Difference vs Ward History: {(rf_all_mae - rf_w_mae):.4f} MAE (positive means worse)')
print('Decision: EXCLUDE municipality context from final model as it degrades held-out performance.')


### Step 4: Feature Importance Ranking & Output Export

In [5]:
importances = pd.DataFrame({
    'Feature': ward_features,
    'Importance': rf_w.feature_importances_
}).sort_values(by='Importance', ascending=False)

print('Feature Importances:')
for _, r in importances.iterrows():
    print(f'  {r["Feature"]:35s}: {r["Importance"]:.4f}')

# Save model artifacts
joblib.dump(rf_w, OUTPUT_DIR / 'random_forest_crosswalk_corrected_model.joblib')
print('Model artifact saved successfully.')
